# Road Accident Severity Analysis for U.S. Data with a Kenya Application Context

**Source dataset:** U.S. road accidents, 2016â€“2023  
**Working dataset in this notebook:** first 50,000 rows from the source CSV  
**Application frame:** Kenyan road-safety planning and evaluation

The original U.S. Accidents file contains approximately 7.7 million records. Because of local memory constraints, this notebook uses a 50,000-row working sample rather than loading the entire dataset into memory. All EDA, statistical analysis, feature engineering, model training, testing, and evaluation in this notebook are based on that 50,000-row sample.

The Kenya context is used to discuss feasibility, data requirements, and transferability. It is not used to claim that U.S. patterns are Kenyan patterns.

The project follows the CRISP-DM sequence: Business Understanding, Data Understanding, Data Preparation, Exploratory Data Analysis, Modeling, Evaluation, Kenya Application / Business Evaluation, and Conclusion.

### Analytical questions
1. Which conditions are associated with more severe accidents?
2. Are there identifiable temporal patterns?
3. What environmental and road characteristics appear alongside severe accidents?
4. Can accident severity be predicted from information available at the time of an incident?
5. What information would a Kenyan road-safety system need?
6. Which U.S. variables would require Kenyan equivalents, and what additional Kenyan data would be required before local deployment?



## 5. Modeling

### Target
We are predicting `high_severity`, defined as `Severity >= 3`.

### Why this target?
This threshold creates a useful operational definition: the model is designed to flag the more serious subset of cases for prioritised review.

### Features
We use variables plausibly known when an accident is reported, including time, weather, visibility, state, and road-context indicators.

### Why these features?
These variables are strongly tied to the accident context and are common in crash-analysis workflows. They can support a decision-support system without using post-event information that would leak the label.

### Train/Test Strategy
We use a chronological split: 80% of the records in time order train the models, and the final 20% form the holdout set. This mirrors the operational problem of forecasting the near future using historical records.

### Baseline
We start with a simple logistic-regression model before comparing more complex tree and boosting models.


In [40]:

model_df = clean_df.sort_values('Start_Time').reset_index(drop=True)
feature_columns = [
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
    'Wind_Speed(mph)', 'hour', 'month', 'is_weekend', 'State',
    'Weather_Condition', 'Sunrise_Sunset', 'day_of_week', 'Amenity',
    'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',
    'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',
    'Turning_Loop'
]
feature_columns = [col for col in feature_columns if col in model_df.columns]

split_index = int(len(model_df) * 0.80)
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

X_train = train_df[feature_columns]
y_train = train_df['high_severity']
X_test = test_df[feature_columns]
y_test = test_df['high_severity']

print(f'Train rows: {len(train_df):,}')
print(f'Test rows: {len(test_df):,}')
print(f'High-severity prevalence in train set: {y_train.mean():.3f}')
print(f'High-severity prevalence in test set: {y_test.mean():.3f}')
print('Model features:', feature_columns)


Train rows: 40,000
Test rows: 10,000
High-severity prevalence in train set: 0.403
High-severity prevalence in test set: 0.375
Model features: ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'hour', 'month', 'is_weekend', 'State', 'Weather_Condition', 'Sunrise_Sunset', 'day_of_week', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']


In [43]:
from sklearn.dummy import DummyClassifier

numeric_features = [
    col for col in feature_columns if col not in [
        'State', 'Weather_Condition', 'Sunrise_Sunset', 'day_of_week', 'Amenity',
        'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',
        'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop'
    ]
]
categorical_features = [col for col in feature_columns if col not in numeric_features]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features)
])

models = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random forest': RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=12,
        class_weight='balanced_subsample',
        n_jobs=1,
        random_state=RANDOM_STATE,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
}

cv = TimeSeriesSplit(n_splits=4)
scoring = {'recall': 'recall', 'precision': 'precision', 'f1': 'f1'}
cv_rows = min(len(X_train), 80_000)
X_cv = X_train.iloc[-cv_rows:]
y_cv = y_train.iloc[-cv_rows:]

cv_results = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    scores = cross_validate(pipe, X_cv, y_cv, cv=cv, scoring=scoring, n_jobs=1)
    cv_results.append({
        'model': name,
        'recall_mean': scores['test_recall'].mean(),
        'precision_mean': scores['test_precision'].mean(),
        'f1_mean': scores['test_f1'].mean(),
    })

cv_df = pd.DataFrame(cv_results).sort_values('recall_mean', ascending=False)
print('Cross-validation summary (mean over time-ordered folds):')
print(cv_df.round(3))


Cross-validation summary (mean over time-ordered folds):
                 model  recall_mean  precision_mean  f1_mean
1  Logistic regression        0.839           0.478    0.606
2        Random forest        0.761           0.504    0.602
3              XGBoost        0.448           0.551    0.475
0             Baseline        0.000           0.000    0.000


In [ ]:

results = []
trained_models = {}

for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    results.append({
        'model': name,
        'recall': recall_score(y_test, pred, zero_division=0),
        'precision': precision_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
        'average_precision': average_precision_score(y_test, prob),
    })
    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values('recall', ascending=False)
print('Holdout test results:')
print(results_df.round(3))


Holdout test results:
                 model  recall  precision     f1  roc_auc  average_precision
1  Logistic regression   0.546      0.465  0.503    0.633              0.484
2        Random forest   0.424      0.507  0.462    0.657              0.498
3              XGBoost   0.157      0.562  0.246    0.657              0.501
0             Baseline   0.000      0.000  0.000    0.500              0.375
